# Hostile Testing Phase 5 - Analytics Connectors (Basic)

## Purpose
Test analytics connector functions with hostile inputs (without actual connections)

## Goal
Add ~10 more functions to reach 5-6% coverage

In [ ]:
import pandas as pd
import tempfile
from pathlib import Path
import shutil

test_results = {
    'function': [],
    'module': [],
    'test_category': [],
    'passed': [],
    'error_message': [],
    'severity': []
}

def record_test(function, module, test_category, passed, error_message="", severity="medium"):
    test_results['function'].append(function)
    test_results['module'].append(module)
    test_results['test_category'].append(test_category)
    test_results['passed'].append(passed)
    test_results['error_message'].append(error_message)
    test_results['severity'].append(severity)

def hostile_test(func, test_name, *args, **kwargs):
    try:
        result = func(*args, **kwargs)
        return (True, result, "")
    except Exception as e:
        return (False, None, str(e))

print("Phase 5 test harness loaded")

## Section 1: Analytics - Google Analytics Profiles

In [ ]:
from siege_utilities import (
    create_ga_account_profile, save_ga_account_profile, load_ga_account_profile
)

test_dir = Path(tempfile.mkdtemp())
try:
    # Test create_ga_account_profile with hostile inputs
    # create_ga_account_profile(account_id, property_id, view_id, client_code, **kwargs)
    hostile_ga_configs = [
        {"account_id": "'; DROP TABLE accounts; --", "property_id": "UA-123", "view_id": "456", "client_code": "TEST"},
        {"account_id": "<script>alert(1)</script>", "property_id": "../../../etc/passwd", "view_id": "789", "client_code": "XSS"},
        {"account_id": "A" * 1000, "property_id": "B" * 1000, "view_id": "C" * 1000, "client_code": "D" * 1000},
        {"account_id": "123\x00456", "property_id": "UA-\x00789", "view_id": "null\x00byte", "client_code": "NULL"},
        {"account_id": "", "property_id": "", "view_id": "", "client_code": ""},
    ]
    
    for cfg in hostile_ga_configs:
        success, result, error = hostile_test(create_ga_account_profile, "hostile_ga", **cfg)
        record_test("create_ga_account_profile", "analytics.google_analytics", "hostile_configs",
                    success, error,
                    "critical" if "DROP TABLE" in str(cfg.get('account_id','')) else "medium")
        
        # If profile created, try to save it
        if success and result:
            success2, result2, error2 = hostile_test(save_ga_account_profile, "save_ga",
                                                     result, str(test_dir))
            record_test("save_ga_account_profile", "analytics.google_analytics", "save_hostile",
                        success2, error2, "medium")
    
    # Test load with hostile client codes
    hostile_codes = [
        None, "", "../../../etc/passwd", "'; DROP TABLE clients; --",
        "<script>alert(1)</script>", "A" * 1000, "code\x00null"
    ]
    
    for code in hostile_codes:
        success, result, error = hostile_test(load_ga_account_profile, "load_ga",
                                              code, str(test_dir))
        record_test("load_ga_account_profile", "analytics.google_analytics", "hostile_codes",
                    success, error,
                    "high" if "../" in str(code) else "medium")
    
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

print(f"Completed {len([r for r in test_results['module'] if 'google' in r])} GA tests")

## Section 2: Analytics - Facebook Business Profiles

In [ ]:
from siege_utilities import (
    create_facebook_account_profile, save_facebook_account_profile, load_facebook_account_profile
)

test_dir = Path(tempfile.mkdtemp())
try:
    # Test create_facebook_account_profile with hostile inputs
    # create_facebook_account_profile(account_id, access_token, client_code, **kwargs)
    hostile_fb_configs = [
        {"account_id": "'; DROP TABLE accounts; --", "access_token": "token123", "client_code": "TEST"},
        {"account_id": "<script>alert(1)</script>", "access_token": "../../../etc/passwd", "client_code": "XSS"},
        {"account_id": "A" * 1000, "access_token": "B" * 1000, "client_code": "C" * 1000},
        {"account_id": "123\x00456", "access_token": "token\x00null", "client_code": "NULL"},
    ]
    
    for cfg in hostile_fb_configs:
        success, result, error = hostile_test(create_facebook_account_profile, "hostile_fb", **cfg)
        record_test("create_facebook_account_profile", "analytics.facebook_business", "hostile_configs",
                    success, error,
                    "critical" if "DROP TABLE" in str(cfg.get('account_id','')) else "medium")
        
        # If profile created, try to save it
        if success and result:
            success2, result2, error2 = hostile_test(save_facebook_account_profile, "save_fb",
                                                     result, str(test_dir))
            record_test("save_facebook_account_profile", "analytics.facebook_business", "save_hostile",
                        success2, error2, "medium")
    
    # Test load with hostile client codes
    for code in hostile_codes:
        success, result, error = hostile_test(load_facebook_account_profile, "load_fb",
                                              code, str(test_dir))
        record_test("load_facebook_account_profile", "analytics.facebook_business", "hostile_codes",
                    success, error,
                    "high" if "../" in str(code) else "medium")
    
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

print(f"Completed {len([r for r in test_results['module'] if 'facebook' in r])} FB tests")

## Section 3: Generate Phase 5 Results

In [ ]:
df = pd.DataFrame(test_results)

print("\n" + "="*80)
print("PHASE 5 HOSTILE TESTING SUMMARY")
print("="*80)
print(f"\nTests run: {len(df)}")
print(f"Passed: {df['passed'].sum()}")
print(f"Failed: {(~df['passed']).sum()}")
print(f"Pass rate: {df['passed'].sum() / len(df) * 100:.1f}%")

unique = df['function'].nunique()
print(f"\nUnique functions: {unique}")
print(f"Phase 5 coverage: {unique}/751 = {unique/751*100:.1f}%")

print("\n" + "="*80)
print("RESULTS BY MODULE")
print("="*80)
summary = df.groupby('module').agg({'passed': ['sum', 'count']})
summary.columns = ['passed', 'total']
summary['pass_rate'] = (summary['passed'] / summary['total'] * 100).round(1)
print(summary)

failures = df[~df['passed']]
if len(failures) > 0:
    print("\n" + "="*80)
    print("FAILURES BY SEVERITY")
    print("="*80)
    print(failures.groupby('severity').size())
    
    critical_high = failures[failures['severity'].isin(['critical', 'high'])]
    if len(critical_high) > 0:
        print("\n" + "="*80)
        print("CRITICAL/HIGH FAILURES (First 5)")
        print("="*80)
        for idx, row in critical_high.head(5).iterrows():
            print(f"\n{row['function']} ({row['severity']})")
            print(f"  Module: {row['module']}")
            print(f"  Error: {row['error_message'][:100]}")
else:
    print("\n✅ NO FAILURES")

df.to_csv("hostile_testing_phase5_results.csv", index=False)
print(f"\nSaved: hostile_testing_phase5_results.csv")

print("\n" + "="*80)
print("FUNCTIONS TESTED")
print("="*80)
for func in df['function'].unique():
    tests = df[df['function'] == func]
    passed = tests['passed'].sum()
    total = len(tests)
    print(f"{func}: {passed}/{total} ({passed/total*100:.0f}%)")

print("\n" + "="*80)
print("CUMULATIVE COVERAGE (Phases 1-5)")
print("="*80)
total_unique = 34 + unique  # Previous phases had 34 unique
print(f"Total unique functions: ~{total_unique}")
print(f"Cumulative coverage: ~{total_unique}/751 = {total_unique/751*100:.1f}%")
print(f"\n🎯 TARGET: 25% coverage (188 functions) - Current: {total_unique}/188")